In [ ]:
import os
print(os.getcwd())   # vs code이용 해당 데이터가 있는 파일 가져오기 +확인
print(os.listdir())  

In [ ]:
import pandas as pd

years = ['2021', '2022', '2023', '2024', '2025']

school_all_df = pd.concat(
    pd.read_excel(
        '/Users/hwangchaewon/공모전/★중앙회 제공데이터/★2021-2025 학교안전사고 데이터.xlsx',
        sheet_name=years
    ).values(),
)

compe_all_df = pd.concat(
    pd.read_excel(
        '/Users/hwangchaewon/공모전/★중앙회 제공데이터/★2021-2025 학교안전사고 보상 데이터.xlsx',
        sheet_name=years
    ).values(),
)

print("학교 안전사고 데이터:", school_all_df.shape)  #정상적으로 로드 되었는지 확인
print("학교 안전사고 보상 데이터:", compe_all_df.shape)

In [ ]:
#학교 안전 사고 데이터와 보상 데이터 열 정보 확인하기
print(school_all_df.columns)
print(compe_all_df.columns)

In [ ]:
print(school_all_df.head()) 
print(compe_all_df.head())

In [ ]:

school_use_df = school_all_df[
    ['학교급','사고자학년','사고자성별', '사고연월', '사고발생시각', '사고요일','사고장소','사고부위','사고형태','사고당시활동']
]

money_cols = ['요양급여', '장해급여', '간병급여', '유족급여', '장례비', '위로금', '보전비용']

# 1. 금액 열 타입 확인
print(compe_all_df[money_cols].dtypes)

# 2. 쉼표, 공백 제거 후 숫자로 변환
for col in money_cols:
    compe_all_df[col] = (
        compe_all_df[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.replace(' ', '', regex=False)
    )
    compe_all_df[col] = pd.to_numeric(compe_all_df[col], errors='coerce').fillna(0)

# 3. 총보상금 만들기 (열 선택 전에 먼저 생성해야 아래 선택에 포함 가능)
compe_all_df['총보상금'] = compe_all_df[money_cols].sum(axis=1)

# 4. 분석에 사용할 열만 선택 (총보상금 포함)
compe_use_df = compe_all_df[
    [
        '학교급',
        '사고자학년',
        '사고자성별',
        '사고시간',
        '사고장소',
        '사고부위',
        '사고형태',
        '사고당시활동',
        '총보상금'
    ]
]

# 5. 확인
print(compe_use_df['총보상금'].head())

#위험도는 보상금들의 합으로 측정하며 가격이 높을 수록 위험도가 높다고 가정한다.

print(school_use_df.columns, school_use_df.shape)
print(compe_use_df.columns, compe_use_df.shape)

In [ ]:
print(school_all_df['학교급'].unique())
print(compe_all_df['학교급'].unique())

In [ ]:
# 나이대를 구분하기위해 사고 ,보상 데이터를 학교별로 분리
school_elem_df = school_use_df[school_use_df['학교급'] == '초등학교']
school_middle_df = school_use_df[school_use_df['학교급'] == '중학교']
school_high_df = school_use_df[school_use_df['학교급'] == '고등학교']
school_pre_df = school_use_df[school_use_df['학교급'] == '유치원']
school_special_df = school_use_df[school_use_df['학교급'] == '특수학교']
school_extra_df = school_use_df[school_use_df['학교급'] == '기타학교']

compe_elem_df = compe_use_df[compe_use_df['학교급'] == '초등학교']
compe_middle_df = compe_use_df[compe_use_df['학교급'] == '중학교']
compe_high_df = compe_use_df[compe_use_df['학교급'] == '고등학교']
compe_pre_df = compe_use_df[compe_use_df['학교급'] == '유치원']
compe_special_df = compe_use_df[compe_use_df['학교급'] == '특수학교']
compe_extra_df = compe_use_df[compe_use_df['학교급'] == '기타학교']

# school_elem_df
#school_middle_df
#school_high_df
#school_pre_df
#school_special_df
#school_extra_df

#compe_elem_df
#compe_middle_df
#compe_high_df
#compe_pre_df
#compe_special_df
#compe_extra_df



# 초등학생 분석 진행

In [ ]:
#초등학생 분석 진행
print(school_elem_df['사고자학년'].unique())
print(compe_elem_df['사고자학년'].unique())

In [ ]:
#유아값 찾음
school_elem_df[school_elem_df['사고자학년'] == '유아']
#초등학교 데이터 중 사고자 학년이 '유아'로 기록된 사례가 3건 존재하였으나,
# 전체 초등학교 사고 데이터 대비 매우 적은 비중을 차지하며 보상 데이터와의 연계 분석이 어려워
# 본 분석에서는 제외하였다.
school_elem_df = school_elem_df[
    school_elem_df['사고자학년'] != '유아'
]

In [ ]:
grade= [ '1학년', '2학년', '3학년', '4학년', '5학년', '6학년']

school_elem_df['사고자학년'] = pd.Categorical(
    school_elem_df['사고자학년'],
    categories=grade,
    ordered=True
)

school_elem_df = school_elem_df.sort_values('사고자학년')

compe_elem_df['사고자학년'] = pd.Categorical(
    compe_elem_df['사고자학년'],
    categories=grade,
    ordered=True
)

compe_elem_df = compe_elem_df.sort_values('사고자학년')



In [ ]:
print(school_elem_df.columns)
print(compe_elem_df.columns)

In [ ]:
school_type_df = (
    school_elem_df
    .groupby(['사고자학년', '사고형태'])
    .size()
    .reset_index(name='사고건수')
    .sort_values(
        ['사고자학년', '사고건수'],
        ascending=[True, False]
    )
)

print(school_type_df.head(20))

In [ ]:
compe_type_df = (
    compe_elem_df
    .groupby(['사고형태', '사고자학년'])
    .agg(
        보상건수=('총보상금', 'count'),
        총보상금=('총보상금', 'sum'),
        평균보상금=('총보상금', 'mean')
    )
    .reset_index()
    .sort_values(
        ['사고형태', '총보상금'],
        ascending=[True, False]
    )
)

print(compe_type_df)

In [ ]:
print(compe_type_df['사고형태'].unique())

# 최종 위험도 분석 (종합위험지수 = 사고건수 × 평균보상금)

In [ ]:
# [요청3] 초등학교 학년 × 사고형태 기준 종합위험지수 (school_type_df + compe_type_df 결합)
risk_df = pd.merge(
    school_type_df,
    compe_type_df,
    on=['사고자학년', '사고형태'],
    how='inner'
)

risk_df['평균보상금'] = (
    risk_df['총보상금']
    / risk_df['보상건수']
)

risk_df['종합위험지수'] = (
    risk_df['사고건수']
    * risk_df['평균보상금']
)

risk_df = risk_df.sort_values(
    '종합위험지수',
    ascending=False
)

print(risk_df.head(10))

# 안전사고 예방 모델: 학교급 + 학년 → 위험 사고유형·장소 추천

In [ ]:
# [요청4] 초·중·고 통합 + (학교급·학년·사고형태·사고장소) 단위 위험도
group_keys = ['학교급', '사고자학년', '사고형태', '사고장소']
target_levels = ['초등학교', '중학교', '고등학교']

# 초·중·고만 사용 (사고자학년이 '유아'인 행은 제외)
school_main_df = school_use_df[
    (school_use_df['학교급'].isin(target_levels)) &
    (school_use_df['사고자학년'] != '유아')
]
compe_main_df = compe_use_df[
    compe_use_df['학교급'].isin(target_levels)
]

# 사고건수 집계
school_group_df = (
    school_main_df
    .groupby(group_keys)
    .size()
    .reset_index(name='사고건수')
)

# 보상 집계
compe_group_df = (
    compe_main_df
    .groupby(group_keys)
    .agg(
        보상건수=('총보상금', 'count'),
        총보상금=('총보상금', 'sum')
    )
    .reset_index()
)

# 위험도 결합
risk_full_df = pd.merge(
    school_group_df,
    compe_group_df,
    on=group_keys,
    how='inner'
)

risk_full_df['평균보상금'] = (
    risk_full_df['총보상금'] / risk_full_df['보상건수']
)
risk_full_df['종합위험지수'] = (
    risk_full_df['사고건수'] * risk_full_df['평균보상금']
)

risk_full_df = risk_full_df.sort_values('종합위험지수', ascending=False)
print(risk_full_df.head(10))

In [ ]:
# [최종 목표] 학교급 + 학년 입력 → 가장 위험한 사고유형·장소 추천
def recommend_risk(학교급, 사고자학년, top_n=5):
    result = risk_full_df[
        (risk_full_df['학교급'] == 학교급) &
        (risk_full_df['사고자학년'] == 사고자학년)
    ].sort_values('종합위험지수', ascending=False)
    return result[
        ['사고형태', '사고장소', '사고건수', '평균보상금', '종합위험지수']
    ].head(top_n)

# 예시: 초등학교 1학년에게 가장 위험한 사고유형·장소 Top 5
print(recommend_risk('초등학교', '1학년'))